In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Cumulative Score").getOrCreate()
df = spark.read.option("header", True).csv("/FileStore/tables/ipl_sample_deliveries.csv")
df.display()

match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,6,BB McCullum,P Kumar,SC Ganguly,1,0,1,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,SC Ganguly,Z Khan,BB McCullum,2,0,2,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,2,SC Ganguly,Z Khan,BB McCullum,0,0,0,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,3,SC Ganguly,Z Khan,BB McCullum,4,0,4,null,0,null,null,null
335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,4,SC Ganguly,Z Khan,BB McCullum,6,0,6,null,0,null,null,null


In [0]:
window_spec = Window.partitionBy("match_id", "inning") \
                    .orderBy(col("over").cast("int"), col("ball").cast("int")) \
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_with_cumulative = df.withColumn("cumulative_score",_sum(col("total_runs").cast("int")).over(window_spec))
df_with_cumulative.select("match_id", "inning", "over", "ball", "total_runs", "cumulative_score").show(20, truncate=False)

+--------+------+----+----+----------+----------------+
|match_id|inning|over|ball|total_runs|cumulative_score|
+--------+------+----+----+----------+----------------+
|335982  |1     |0   |1   |1         |1               |
|335982  |1     |0   |2   |0         |1               |
|335982  |1     |0   |3   |1         |2               |
|335982  |1     |0   |4   |0         |2               |
|335982  |1     |0   |5   |0         |2               |
|335982  |1     |0   |6   |1         |3               |
|335982  |1     |1   |1   |2         |5               |
|335982  |1     |1   |2   |0         |5               |
|335982  |1     |1   |3   |4         |9               |
|335982  |1     |1   |4   |6         |15              |
|335982  |1     |1   |5   |1         |16              |
|335982  |1     |1   |6   |3         |19              |
|335982  |1     |2   |1   |0         |19              |
|335982  |1     |2   |2   |1         |20              |
|335982  |1     |2   |3   |1         |21        